In [4]:
import io
import requests
from bs4 import BeautifulSoup
import pandas as pd
import yfinance as yf
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Fetch the HTML content with headers
response = requests.get(url, headers=headers)

# Check if the request was successful (status code 200)
if response.status_code == 200:
    # Parse the HTML content of the webpage
    soup = BeautifulSoup(response.content, "html.parser")

    # Find the S&P 500 table directly by its id "constituents"
    table = soup.find("table", id="constituents")

    # Use pandas to read the table into a DataFrame
    df = pd.read_html(io.StringIO(str(table)))[0]

    # print(df)

    # # Extract year from the addition date
    df['Year Added'] = pd.to_datetime(df['Date added']).dt.year
    
    
   


    # # 1. Create DataFrame with company tickers, names, and the year they were added
    companies_df = df[['Symbol', 'Security', 'Year Added']].copy()
    companies_df.rename(columns={'Symbol': 'Ticker', 'Security': 'Company Name'}, inplace=True)
    # print("--- Companies DataFrame (Sample) ---")
    # print(companies_df.head(10))

    # # 2. Calculate the number of stocks added each year
    additions_per_year = df['Year Added'].value_counts().sort_index()
    # print("\n--- Number of Stocks Added Each Year ---")
    # print(additions_per_year)

    # # 3. Determine which full year had the highest number of additions starting from 2020
    # Full completed years starting from 2020 up to 2025
    additions_from_2020 = additions_per_year[additions_per_year.index >= 2020]
    # print("\n--- Additions starting from 2020 ---")
    # print(additions_from_2020)

    # # Filter for full years (completed calendar years 2020-2025)
    full_years = additions_per_year[(additions_per_year.index >= 2020) & (additions_per_year.index < 2026)]
    print(full_years)
    max_full_year = full_years.idxmax()
    max_additions = full_years.max()

    print(f"\nHighest number of additions in a full year starting from 2020: {max_full_year} ({max_additions} additions)")
else:
    print("Failed to retrieve data from the webpage.")

# task 2===

import yfinance as yf
import pandas as pd

stock = dict()

stockList = [
    ["United States - S&P 500", '^GSPC'],
    ["China - Shanghai Composite", '000001.SS'],
    ["Hong Kong - HANG SENG INDEX", '^HSI'],
    ["Australia - S&P/ASX 200", '^AXJO'],
    ["India - Nifty 50", '^NSEI'],
    ["Canada - S&P/TSX Composite", '^GSPTSE'],
    ["Germany - DAX", '^GDAXI'],
    ["United Kingdom - FTSE 100", '^FTSE'],
    ["Japan - Nikkei 225", '^N225'],
    ["Mexico - IPC Mexico", '^MXX'],
    ["Brazil - Ibovespa", '^BVSP'],
]

# Step 1: download daily history for each index and store it
for name, ticker in stockList:
    df = yf.Ticker(ticker).history(start="2026-01-01", end="2026-08-21", interval="1d")
    stock[name] = df

# Step 2: compute YTD return for each index
results = []
for name, ticker in stockList:
    df = stock[name].dropna()          # drop rows with missing close (holidays etc.)
    if df.empty:
        print(f"No data for {name}")
        continue
    start_price = df['Close'].iloc[0]  # first trading day close in the window
    end_price   = df['Close'].iloc[-1] # last trading day close in the window
    ytd_return = (end_price / start_price - 1) * 100
    results.append((name, ytd_return))

# Step 3: put into a DataFrame and compare against the US
# print(results)
returns_df = pd.DataFrame(results, columns=["Index", "YTD Return (%)"])
returns_df = returns_df.sort_values("YTD Return (%)", ascending=False).reset_index(drop=True)

# print(returns_df)
us_return = returns_df.loc[returns_df["Index"] == "United States - S&P 500", "YTD Return (%)"].values[0]

# print(us_return)

better_than_us = returns_df[
    (returns_df["Index"] != "United States - S&P 500") &
    (returns_df["YTD Return (%)"] > us_return)
]

# print(returns_df)
print(f"\nS&P 500 YTD return: {us_return:.2f}%")
print(f"Number of indexes (out of {len(stockList)-1}) beating the US: {len(better_than_us)}")
print(better_than_us)

Year Added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
Name: count, dtype: int64

Highest number of additions in a full year starting from 2020: 2025 (18 additions)

S&P 500 YTD return: 11.41%
Number of indexes (out of 10) beating the US: 2
                        Index  YTD Return (%)
0          Japan - Nikkei 225       27.750745
1  Canada - S&P/TSX Composite       14.057466
